In [5]:
import pandas as pd
import numpy as np

from week02.day1_conditions_regex import category

# ---Loading my transcriptome data---
df = pd.read_csv("./my_transcriptome_data/gene_fpkm.csv")

expr_cols = ['R1','R2','R3',
             'CO1','CO2','CO3',
             'CT1','CT2','CT3',
             'CR1','CR2','CR3']

expr_df = df[expr_cols]

# ── Multi-condition filtering: Identify highly expressed genes ──────────────────────────
# Condition 1: Mean across all samples > 1 (filter out genes with negligible expression)
# Condition 2: CR group mean > R group mean (upregulated under cold stress)
all_mean = expr_df.mean(axis=1)
cr_mean = expr_df[['CR1','CR2','CR3']].mean(axis=1)
r_mean = expr_df[['R1','R2','R3']].mean(axis=1)
filtered = df[
    (all_mean > 1) &
    (all_mean < cr_mean)
]

print(f"total gene number: {len(df)}")
print(f"Highly expressed gene number: {len(filtered)}")
print(f"\nThe first 5 line:")
display(filtered[['gene_id']+expr_cols].head())

total gene number: 123202
Highly expressed gene number: 11463

The first 5 line:


,gene_id,R1,R2,R3,CO1,CO2,CO3,CT1,CT2,CT3,CR1,CR2,CR3
3,Cluster-14358.55088,0.00,0.14,0.03,4.24,0.06,1.53,1.32,0.27,0.90,3.70,0.27,0.00
18,Cluster-14358.37655,83.17,70.10,108.20,96.09,43.71,94.82,51.90,100.44,97.94,113.85,96.12,141.98
30,Cluster-14358.43937,5.53,2.91,5.28,9.86,12.83,9.79,12.73,6.60,11.87,14.39,10.27,10.47
34,Cluster-14358.22263,9.58,10.99,8.87,16.13,10.60,13.46,11.79,11.33,8.31,13.35,12.14,12.58
37,Cluster-14358.29908,0.00,0.00,0.00,2.08,2.82,9.18,7.53,3.97,2.41,5.00,3.50,16.64


In [16]:
# ---Merge expression data with annotation information---
anno_cols = ['gene_id', 'NR Description', 'BP Description',
             'MF Description', 'CC Description']
anno_df = df[anno_cols]

# Merge filtered genes with annotation information
merged = filtered[['gene_id'] + expr_cols].merge(anno_df, on='gene_id')
print(f"The shape after merged: {merged.shape}")
display(merged[['gene_id','NR Description','BP Description']].head())

# ---Remove no annotated genes---
has_nr = merged[merged['NR Description'] != '--']
print(f"\nGenes with NR annotation: {len(has_nr)}")
print(f"\nGenes without NR annotation: {len(merged) - len(has_nr)}")

# ---groupby: Summerize by BP funtional category---
# BP Description = Biological Process
# Extract the first key word of BP annotation
has_bp = merged[merged['BP Description'] != '--'].copy()
has_bp['bp_short'] = has_bp['BP Description'].str[:40]

bp_count = has_bp.groupby('bp_short').agg(
    gene_count=('gene_id', 'count')).reset_index().sort_values('gene_count', ascending=False)

print(f"\nBiological Process(first 10)")
display((bp_count.head(10)))

The shape after merged: (11463, 17)


,gene_id,NR Description,BP Description
0,Cluster-14358.55088,"protoporphyrinogen oxidase, mitochondrial-like...",oxidation-reduction process
1,Cluster-14358.37655,hypothetical protein MA16_Dca009951 [Dendrobiu...,--
2,Cluster-14358.43937,Calcineurin B-like protein 1 [Dendrobium caten...,--
3,Cluster-14358.22263,probable serine/threonine-protein kinase PBL7 ...,"positive regulation of transcription, DNA-temp..."
4,Cluster-14358.29908,hypothetical protein MA16_Dca011472 [Dendrobiu...,--



Genes with NR annotation: 8854

Genes without NR annotation: 2609

Biological Process(first 10)


,bp_short,gene_count
1198,"regulation of transcription, DNA-templat",307
1003,protein phosphorylation,222
771,oxidation-reduction process,188
1431,transmembrane transport,96
1247,ribosome biogenesis//translation,93
1417,translation//ribosome biogenesis,87
1057,protein ubiquitination,78
223,carbohydrate metabolic process,59
1075,proteolysis,55
1267,signal transduction,36


In [18]:
# ---apply: Assign expression pattern labels to each gene---

# Calculate group mean, add to merged matrix
merged['R_mean'] = merged[['R1','R2','R3']].mean(axis=1)
merged['CO_mean'] = merged[['CO1','CO2','CO3']].mean(axis=1)
merged['CT_mean'] = merged[['CT1','CT2','CT3']].mean(axis=1)
merged['CR_mean'] = merged[['CR1','CR2','CR3']].mean(axis=1)

# Tag each gene with apply
def classify_expression(row):
    """Seperate genes into 4 categories based on gene expression patterns"""
    r = row['R_mean']
    cr = row['CR_mean']

    if r < 1 and cr>5:
        return 'Cold-induced'    # Cold inducted (R nearly no expression)
    elif r > 5 and cr < 1:
        return 'Up-repressed'    #Cold repress
    elif cr > r * 2:
        return 'Down-regulated'  #Down regulation
    else:
        return 'Stable'          # Stable expression

merged['expression_pattern'] = merged.apply(classify_expression, axis=1)

# Calculate the number of each gene category
pattern_counts = merged["expression_pattern"].value_counts()
print(f"\nExpression pattern categories:")
print(pattern_counts)


Expression pattern categories:
expression_pattern
Down-regulated    5811
Stable            4534
Cold-induced      1118
Name: count, dtype: int64
